# **Persiapan Library dan Data**

In [149]:
from google.colab import drive
drive.mount('/content/drive')
# Path ke folder di Google Drive
folder_path = '/content/drive/MyDrive/Text Analysis LSTM'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [150]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [151]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers

In [152]:
file_path = '/content/drive/MyDrive/Text Analysis LSTM/train.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1')

In [153]:
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [154]:
print(df.shape)
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.info)

(27481, 10)
textID              0
text                1
selected_text       1
sentiment           0
Time of Tweet       0
Age of User         0
Country             0
Population -2020    0
Land Area (Km²)     0
Density (P/Km²)     0
dtype: int64
0
<bound method DataFrame.info of            textID                                               text  \
0      cb774db0d1                I`d have responded, if I were going   
1      549e992a42      Sooo SAD I will miss you here in San Diego!!!   
2      088c60f138                          my boss is bullying me...   
3      9642c003ef                     what interview! leave me alone   
4      358bd9e861   Sons of ****, why couldn`t they put them on t...   
...           ...                                                ...   
27476  4eac33d1c0   wish we could come see u on Denver  husband l...   
27477  4f4c4fc327   I`ve wondered about rake to.  The client has ...   
27478  f67aae2310   Yay good for both of you. Enjoy the break - y...   
2

In [155]:
df['sentiment'].value_counts()

sentiment
neutral     11118
positive     8582
negative     7781
Name: count, dtype: int64

In [156]:
df.dropna(inplace = True)

In [157]:
df.head()

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18
3,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470.0,164
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative,noon,60-70,Angola,32866272,1246700.0,26


In [158]:
for i in range(10):
    print(df['text'][i+1])

 Sooo SAD I will miss you here in San Diego!!!
my boss is bullying me...
 what interview! leave me alone
 Sons of ****, why couldn`t they put them on the releases we already bought
http://www.dothebouncy.com/smf - some shameless plugging for the best Rangers forum on earth
2am feedings for the baby are fun when he is all smiles and coos
Soooo high
 Both of you
 Journey!? Wow... u just became cooler.  hehe... (is that possible!?)
 as much as i love to be hopeful, i reckon the chances are minimal =P i`m never gonna get my cake and stuff


# **Preprocessing Text**

In [159]:
import re
def clean(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Conver to lower
    text = text.lower()
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [160]:
df['text'] = df['text'].apply(clean)

In [161]:
for i in range(10):
    print(df['text'][i+1])

sooo sad i will miss you here in san diego
my boss is bullying me
what interview leave me alone
sons of why couldnt they put them on the releases we already bought
httpwwwdothebouncycomsmf some shameless plugging for the best rangers forum on earth
am feedings for the baby are fun when he is all smiles and coos
soooo high
both of you
journey wow u just became cooler hehe is that possible
as much as i love to be hopeful i reckon the chances are minimal p im never gonna get my cake and stuff


In [162]:
df.shape

(27480, 10)

# **Visualize Data**

In [163]:
cyberpunk_palette = ["#FF00FF", "#00FF00", "#0000FF"]  # Neon pink, green, and blue
template = "plotly_dark"

In [164]:
import plotly.express as px
import plotly.graph_objs as go
from wordcloud import WordCloud

In [165]:
import plotly.graph_objs as go
import plotly.colors as colors
import numpy as np

# Word Length Distribution
word_lengths = [len(word) for text in df['text'] for word in text.split()]
word_lengths_counts = {length: word_lengths.count(length) for length in set(word_lengths)}

# Sort the word lengths by count in descending order
sorted_word_lengths = sorted(word_lengths_counts.items(), key=lambda x: x[1], reverse=True)

# Create a custom colorscale with a fixed number of colors
colorscale = colors.sample_colorscale('Viridis', len(word_lengths_counts))

# Create the bar chart trace
bar_trace = go.Bar(
    x=[length for length, count in sorted_word_lengths],
    y=[count for length, count in sorted_word_lengths],
    marker=dict(
        color=[colorscale[i] for i in range(len(sorted_word_lengths))],
        line=dict(
            color=cyberpunk_palette[0],
            width=2
                    )
    ),
    hovertemplate='Word Length: %{x}<br>Count: %{y}<extra></extra>'
)

# Create the neon light effect trace
light_effect_trace = go.Scatter(
    x=[length for length, count in sorted_word_lengths],
    y=[count * 1.05 for length, count in sorted_word_lengths],
    mode='lines',
    line=dict(
        color=cyberpunk_palette[1],
        width=5
    ),
    hoverinfo='skip'
)

# Create the layout
layout = go.Layout(
    title="Word Length Distribution",
    xaxis=dict(
        title="Word Length",
        tickfont=dict(color=cyberpunk_palette[2])
    ),
        yaxis=dict(
        title="Count",
        tickfont=dict(color=cyberpunk_palette[2])
    ),
    plot_bgcolor="black",
    paper_bgcolor="black",
    font_color=cyberpunk_palette[2],
    title_font_color=cyberpunk_palette[2],
    title_font_size=20,
    margin=dict(t=80, l=100, r=50, b=100)
)

# Create the figure and show it
fig = go.Figure(data=[bar_trace, light_effect_trace], layout=layout)
fig.show()

In [166]:
# Sentence Length Distribution
sentence_lengths = [len(text.split()) for text in df['text']]

# Create a custom colorscale with a fixed number of colors
colorscale = colors.sample_colorscale('Viridis', len(set(sentence_lengths)))

bar_trace = go.Bar(
    x=sorted(set(sentence_lengths)),
    y=[sentence_lengths.count(length) for length in sorted(set(sentence_lengths))],
    marker=dict(
        color=[colorscale[i] for i in range(len(set(sentence_lengths)))],
        line=dict(
            color=cyberpunk_palette[0],
            width=2
        )
    ),
    hovertemplate='Sentence Length: %{x}<br>Count: %{y}<extra></extra>'
)

light_effect_trace = go.Scatter(
    x=sorted(set(sentence_lengths)),
    y=[sentence_lengths.count(length) * 1.05 for length in sorted(set(sentence_lengths))],
    mode='lines',
    line=dict(
        color=cyberpunk_palette[1],
        width=5
    ),
    hoverinfo='skip'
)

layout = go.Layout(
    title="Sentence Length Distribution",
    xaxis=dict(
        title="Sentence Length",
        tickfont=dict(color=cyberpunk_palette[2])
    ),
    yaxis=dict(
        title="Count",
        tickfont=dict(color=cyberpunk_palette[2])
    ),
    plot_bgcolor="black",
    paper_bgcolor="black",
    font_color=cyberpunk_palette[2],
    title_font_color=cyberpunk_palette[2],
    title_font_size=20,
    margin=dict(t=80, l=100, r=50, b=100)
)

fig = go.Figure(data=[bar_trace, light_effect_trace], layout=layout)
fig.show()

In [167]:
# Word Cloud for Positive Sentiment
positive_text = ' '.join(df[df['sentiment'] == 'positive']['text'])
wordcloud = WordCloud(background_color='black', width = 800, height = 400, max_words=200, colormap='Greens').generate(positive_text)
fig = go.Figure(go.Image(z = np.dstack((wordcloud.to_array(), wordcloud.to_array(), wordcloud.to_array()))))
fig.update_layout(
    title = 'Word Cloud For Positive Sentiment',
    template = template,
    plot_bgcolor = 'black',
    paper_bgcolor = 'black',
    font_color = cyberpunk_palette[2],
    title_font_color = cyberpunk_palette[2],
    title_font_size = 20,
    margin = dict(t = 80, l = 50, r = 50, b = 50)
)
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [168]:
# Word Cloud for Negative Sentiment
negative_text = ' '.join(df[df['sentiment'] == 'negative']['text'])
wordcloud = WordCloud(background_color='black', width=800, height=400, max_words=200, colormap='Reds').generate(negative_text)
fig = go.Figure(go.Image(z=np.dstack((wordcloud.to_array(), wordcloud.to_array(), wordcloud.to_array()))))
fig.update_layout(
    title="Word Cloud for Negative Sentiment",
    template=template,
    plot_bgcolor="black",
    paper_bgcolor="black",
    font_color=cyberpunk_palette[2],
    title_font_color=cyberpunk_palette[2],
    title_font_size=20,
    margin=dict(t=80, l=50, r=50, b=50)
)
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [169]:
# Word Cloud for Neutral Sentiment
neutral_text = ' '.join(df[df['sentiment'] == 'neutral']['text'])
wordcloud = WordCloud(background_color='black', width=800, height=400, max_words=200, colormap='Blues').generate(neutral_text)
fig = go.Figure(go.Image(z=np.dstack((wordcloud.to_array(), wordcloud.to_array(), wordcloud.to_array()))))
fig.update_layout(
    title="Word Cloud for Neutral Sentiment",
    template=template,
    plot_bgcolor="black",
    paper_bgcolor="black",
    font_color=cyberpunk_palette[2],
    title_font_color=cyberpunk_palette[2],
    title_font_size=20,
    margin=dict(t=80, l=50, r=50, b=50)
)
fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [170]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# **Tokenize Text**

In [171]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])

# **Padding The Sequences**

In [172]:
max_length = max([len(seq) for seq in sequences])
padded_sequences = pad_sequences(sequences, maxlen = max_length, padding = 'post')

# **Prepare The Target Variable**

In [173]:
labels = pd.get_dummies(df['sentiment']).values

In [174]:
xtrian,xtest,ytrain,ytest = train_test_split(padded_sequences, labels, test_size = 0.1)

# **Arsitektur Model**

In [175]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, BatchNormalization,RNN
# from kerastuner.tuners import RandomSearch
from sklearn.model_selection import train_test_split

In [176]:
from keras.models import Sequential
from keras.layers import Embedding, LSTM, BatchNormalization, Dense, Dropout
from keras.optimizers import Adam

vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 128
max_length = 32

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim))  # Layer 1: Embedding

model.add(LSTM(32, return_sequences=True))  # Layer 2: LSTM (Pertama)
model.add(BatchNormalization())  # Layer 3: BatchNormalization (Pertama)

model.add(LSTM(64, return_sequences=True))  # Layer 4: LSTM (Kedua)
model.add(BatchNormalization())  # Layer 5: BatchNormalization (Kedua)

model.add(LSTM(64, return_sequences=True))  # Layer 6: LSTM (Ketiga)
model.add(BatchNormalization())  # Layer 7: BatchNormalization (Ketiga)

model.add(LSTM(64))  # Layer 8: LSTM (Keempat)
model.add(BatchNormalization())  # Layer 9: BatchNormalization (Keempat)

model.add(Dense(64, activation='relu'))  # Layer 10: Dense (Pertama)
model.add(Dropout(0.2))  # Layer 11: Dropout (Pertama)

model.add(Dense(64, activation='relu'))  # Layer 12: Dense (Pertama)
model.add(Dropout(0.2))  # Layer 13: Dropout (Pertama)

model.add(Dense(64, activation='relu'))  # Layer 14: Dense (Pertama)
model.add(Dropout(0.2))  # Layer 15: Dropout (Pertama)

model.add(Dense(64, activation='relu'))  # Layer 16: Dense (Kedua)

model.add(Dense(3, activation='softmax'))  # Layer 17: Dense (Ketiga)

model.compile(loss='categorical_crossentropy', optimizer=Adam(), metrics=['accuracy'])

In [177]:
model.fit(xtrian, ytrain, epochs=10, validation_data=(xtest, ytest))

Epoch 1/10
773/773 [==============================] - 101s 120ms/step - loss: 0.9131 - accuracy: 0.5617 - val_loss: 0.7655 - val_accuracy: 0.6790
Epoch 2/10
773/773 [==============================] - 93s 120ms/step - loss: 0.6644 - accuracy: 0.7292 - val_loss: 0.7395 - val_accuracy: 0.7067
Epoch 3/10
773/773 [==============================] - 93s 120ms/step - loss: 0.5244 - accuracy: 0.7966 - val_loss: 0.7492 - val_accuracy: 0.6856
Epoch 4/10
773/773 [==============================] - 92s 119ms/step - loss: 0.4148 - accuracy: 0.8460 - val_loss: 0.7601 - val_accuracy: 0.7020
Epoch 5/10
773/773 [==============================] - 96s 124ms/step - loss: 0.3302 - accuracy: 0.8812 - val_loss: 0.8644 - val_accuracy: 0.6903
Epoch 6/10
773/773 [==============================] - 93s 120ms/step - loss: 0.2664 - accuracy: 0.9085 - val_loss: 0.8603 - val_accuracy: 0.6921
Epoch 7/10
773/773 [==============================] - 91s 118ms/step - loss: 0.2227 - accuracy: 0.9227 - val_loss: 0.8856 - val_a

In [178]:
loss, accuracy = model.evaluate(xtest, ytest)
print("Test Accuracy:", accuracy)

86/86 [==============================] - 2s 25ms/step - loss: 1.0647 - accuracy: 0.6892
Test Accuracy: 0.6892285346984863


In [179]:
model.save('lstm.h5')

/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning:

You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.



In [180]:
model.save('my_model.keras')

In [181]:
import os

# Path lengkap untuk menyimpan model
model_path_keras = os.path.join(folder_path, 'my_model.keras')
model_path_h5 = os.path.join(folder_path, 'lstm.h5')

# Menyimpan model dalam format Keras
model.save(model_path_keras)

# Menyimpan model dalam format HDF5
model.save(model_path_h5)

In [182]:
from tensorflow.keras.models import load_model

# Memuat model dari file
model_save = load_model('/content/drive/MyDrive/Text Analysis LSTM/lstm.h5')  # Atau .keras jika Anda menggunakan format itu

# Mengevaluasi model
loss, accuracy = model.evaluate(xtest, ytest)
print("Test Accuracy:", accuracy)


86/86 [==============================] - 2s 20ms/step - loss: 1.0647 - accuracy: 0.6892
Test Accuracy: 0.6892285346984863


In [183]:
print(max_length)  # Tidak perlu len()

32


In [184]:
def predict_sentiment(input_text, tokenizer, model_save, max_length):
    # Preprocess the input text
    input_sequence = tokenizer.texts_to_sequences([input_text])
    padded_input_sequence = pad_sequences(input_sequence, maxlen=max_length, padding='post')

    # Get the prediction
    prediction = model_save.predict(padded_input_sequence)

    # Convert the prediction to sentiment label
    sentiment_labels = ['Negative', 'Neutral', 'Positive']
    predicted_label_index = np.argmax(prediction)
    predicted_sentiment = sentiment_labels[predicted_label_index]

    return predicted_sentiment

In [185]:
positive_rows = df[df['sentiment'] == 'negative']
print(positive_rows[['text']].head(5))

                                                 text
1          sooo sad i will miss you here in san diego
2                              my boss is bullying me
3                       what interview leave me alone
4   sons of why couldnt they put them on the relea...
12       my sharpie is running dangerously low on ink


In [186]:
input_text = "please leave me alone, im very sad right now"
predicted_sentiment = predict_sentiment(input_text, tokenizer, model_save, max_length)
print("Predicted Sentiment:", predicted_sentiment)

1/1 [==============================] - 2s 2s/step
Predicted Sentiment: Negative


In [187]:
!pip install googletrans==4.0.0-rc1
!pip install transformers tensorflow
from googletrans import Translator
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf

In [188]:
def translate_text(text, src_lang='id', dest_lang='en'):
    translator = Translator()
    translated = translator.translate(text, src=src_lang, dest=dest_lang)
    return translated.text

def preprocess_text(text, tokenizer, max_length=128):
    inputs = tokenizer(text, return_tensors="tf", truncation=True, padding='max_length', max_length=max_length)
    input_ids = inputs['input_ids']
    return input_ids

def predict_sentiment(text, tokenizer, model_save, max_length=128):
    input_ids = preprocess_text(text, tokenizer, max_length)

    # Prediksi sentimen menggunakan model LSTM
    predictions = model_save.predict(input_ids)
    predicted_class = tf.argmax(predictions, axis=1).numpy()[0]

    sentiment_labels = ['Negative', 'Neutral', 'Positive']
    return sentiment_labels[predicted_class]

In [189]:
folder_path = '/content/drive/MyDrive/Text Analysis LSTM'
model_save_path = os.path.join(folder_path, 'lstm_translated.h5')

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Cek apakah file model ada
if os.path.exists(model_save_path):
    # Load existing LSTM model
    model_translated = load_model(model_save_path)
else:
    # Contoh pembuatan dan pelatihan model LSTM
    # Buat model LSTM baru jika belum ada model yang disimpan
    model_translated = tf.keras.Sequential([
        tf.keras.layers.Embedding(input_dim=30522, output_dim=128, input_length=128),
        tf.keras.layers.LSTM(128, return_sequences=False),
        tf.keras.layers.Dense(3, activation='softmax')
    ])
    model_translated.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Simpan model yang baru dilatih
    model_translated.save(model_save_path)

In [190]:
# Input teks dari pengguna
input_text = "tolong tinggalkan aku sendiri, aku sangat sedih sekarang"

# Terjemahkan teks dari Bahasa Indonesia ke Bahasa Inggris
translated_text = translate_text(input_text)

# Prediksi sentimen dari teks yang telah diterjemahkan
predicted_sentiment = predict_sentiment(translated_text, tokenizer, model_translated)

print("Input Text:", input_text)
print("Translated Text:", translated_text)
print("Predicted Sentiment:", predicted_sentiment)


1/1 [==============================] - 0s 499ms/step
Input Text: tolong tinggalkan aku sendiri, aku sangat sedih sekarang
Translated Text: Please leave me alone, I'm very sad now
Predicted Sentiment: Positive
